# Módulo 4 — Preparação dos Dados com Python

In [111]:
## Bibliotecas principais do Módulo 4
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
import unicodedata

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)


In [112]:
# Estrutura padrão do projeto
RAIZ = Path.cwd().parent.resolve()

PASTAS = ["dados_brutos", "dados_tratados", "docs",
 "notebooks", "sql", "dashboards", "resultados",
 "relatorios", "apresentacao", "logs"]

for pasta in PASTAS:
 (RAIZ / pasta).mkdir(
 parents=True, exist_ok=True)

print("Pastas verificadas/criadas:")
for pasta in PASTAS:
 print("-", RAIZ / pasta)

Pastas verificadas/criadas:
- C:\Users\Aluno\Documents\Felipe\FAP-2026-AnaliseDados\Projeto_PRF\dados_brutos
- C:\Users\Aluno\Documents\Felipe\FAP-2026-AnaliseDados\Projeto_PRF\dados_tratados
- C:\Users\Aluno\Documents\Felipe\FAP-2026-AnaliseDados\Projeto_PRF\docs
- C:\Users\Aluno\Documents\Felipe\FAP-2026-AnaliseDados\Projeto_PRF\notebooks
- C:\Users\Aluno\Documents\Felipe\FAP-2026-AnaliseDados\Projeto_PRF\sql
- C:\Users\Aluno\Documents\Felipe\FAP-2026-AnaliseDados\Projeto_PRF\dashboards
- C:\Users\Aluno\Documents\Felipe\FAP-2026-AnaliseDados\Projeto_PRF\resultados
- C:\Users\Aluno\Documents\Felipe\FAP-2026-AnaliseDados\Projeto_PRF\relatorios
- C:\Users\Aluno\Documents\Felipe\FAP-2026-AnaliseDados\Projeto_PRF\apresentacao
- C:\Users\Aluno\Documents\Felipe\FAP-2026-AnaliseDados\Projeto_PRF\logs


In [113]:
RAIZ = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

ARQUIVO_BRUTO = RAIZ / "dados_brutos" / "acidentes2025.csv"
ARQUIVO_BASE_ANALITICA = RAIZ / "dados_tratados" / "base_analitica_prf_2025.csv"
ARQUIVO_BASE_MODELAVEL = RAIZ / "dados_tratados" / "base_modelavel_prf_2025.csv"
ARQUIVO_DICIONARIO = RAIZ / "dados_tratados" / "dicionario_variaveis_modulo4.csv"
ARQUIVO_DECISOES = RAIZ / "logs" / "decisoes_tratamento_modulo4.md"
ARQUIVO_README = RAIZ / "docs" / "README.md"

SEPARADOR = ";"
ENCODING_ENTRADA = "latin1"
ENCODING_SAIDA = "utf-8-sig"

In [114]:
def ler_csv_prf(caminho, sep=";",
 encodings=("latin1","utf-8","utf-8-sig")):
 ultimo_erro = None
 for enc in encodings:
    try:
        print(f"Tentando encoding={enc}...")
        return pd.read_csv(
        caminho, sep=sep,
        encoding=enc, low_memory=False)
    except Exception as erro:
        ultimo_erro = erro
        print(f"Falhou com {enc}: {erro}")
        raise ultimo_erro

df = ler_csv_prf(ARQUIVO_BRUTO, sep=SEPARADOR)
df.head()


Tentando encoding=latin1...


,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,pessoas,mortos,feridos_leves,feridos_graves,ilesos,ignorados,feridos,veiculos,latitude,longitude,regional,delegacia,uop
0,652493,01/01/2025,quarta-feira,06:20:00,SP,116,225,GUARULHOS,Reação tardia ou ineficiente do condutor,Tombamento,Com Vítimas Feridas,Pleno dia,Decrescente,Céu Claro,Múltipla,Reta;Declive,Sim,2,0,1,0,0,1,1,2,"-23,48586772","-46,54075317",SPRF-SP,DEL01-SP,UOP01-DEL01-SP
1,652519,01/01/2025,quarta-feira,07:50:00,CE,116,"546,2",PENAFORTE,Pista esburacada,Colisão frontal,NaN,Pleno dia,Crescente,Céu Claro,Simples,Reta,Não,6,1,1,0,1,4,1,6,"-7,812288","-39,08333306",SPRF-CE,DEL05-CE,UOP03-DEL05-CE
2,652522,01/01/2025,quarta-feira,08:45:00,PR,369,"88,2",CORNELIO PROCOPIO,Reação tardia ou ineficiente do condutor,Colisão traseira,Com Vítimas Feridas,Pleno dia,Crescente,Sol,Dupla,Reta;Aclive,Sim,5,0,3,0,2,0,3,2,"-23,182565","-50,637228",SPRF-PR,DEL07-PR,UOP05-DEL07-PR
3,652544,01/01/2025,quarta-feira,11:00:00,PR,116,74,CAMPINA GRANDE DO SUL,Reação tardia ou ineficiente do condutor,Saída de leito carroçável,Com Vítimas Feridas,Pleno dia,Crescente,Céu Claro,Dupla,Reta,Não,5,0,1,0,4,0,1,2,"-25,36517687","-49,04223028",SPRF-PR,DEL01-PR,UOP02-DEL01-PR
4,652549,01/01/2025,quarta-feira,09:30:00,MG,251,471,FRANCISCO SA,Velocidade Incompatível,Colisão frontal,Com Vítimas Feridas,Pleno dia,Decrescente,Chuva,Simples,Curva;Declive,Não,5,0,1,1,1,2,2,4,"-16,46801304","-43,43121303",SPRF-MG,DEL12-MG,UOP01-DEL12-MG


In [115]:
def normalizar_nome_coluna(nome):
    nome = str(nome).strip().lower()
    nome = unicodedata.normalize(
        "NFKD", nome
    ).encode("ascii","ignore").decode("utf-8")
    nome = nome.replace(" ","_"
        ).replace("-","_").replace("/","_")
    while "__" in nome:
        nome = nome.replace("__","_")
    return nome.strip("_")

df.columns = [normalizar_nome_coluna(c)
              for c in df.columns]
renomear = {
  "condicao_meteorologica":
  "condicao_metereologica"}
df = df.rename(columns={
    k:v for k,v in renomear.items()
    if k in df.columns})

In [116]:
colunas_esperadas = [
 "data_inversa","dia_semana","horario",
 "uf","br","municipio","causa_acidente",
 "tipo_acidente","classificacao_acidente",
 "fase_dia","condicao_metereologica",
 "tipo_pista","tracado_via","uso_solo",
 "pessoas","mortos","feridos_leves",
 "feridos_graves","feridos","veiculos"
]

faltantes = [c for c in colunas_esperadas
    if c not in df.columns]
print("Colunas faltantes:", faltantes)

if faltantes:
    print("Atenção: ajuste nomes ou confirme o dicionário da PRF.")

Colunas faltantes: []


In [117]:
# Tipos de dados e memória utilizada df.info(memory_usage="deep")

resumo_tipos = (
df.dtypes.astype(str)
.value_counts()
.rename_axis("tipo")
.reset_index(name="qtd_colunas"))

display(resumo_tipos)

,tipo,qtd_colunas
0,str,20
1,int64,10


In [118]:
# Diagnóstico de valores ausentes

nulos = pd.DataFrame({
    "qtd_nulos": df.isna().sum(),
    "perc_nulos": df.isna().mean() * 100
}).sort_values(
    "perc_nulos", ascending=False)
display(nulos[nulos["qtd_nulos"] > 0])

,qtd_nulos,perc_nulos
uop,38,0.052393
delegacia,22,0.030333
regional,2,0.002758
classificacao_acidente,1,0.001379


In [119]:
# Diagnóstico e remoção de duplicidades

qtd_duplicadas = df.duplicated().sum()
print("Duplicidades exatas:", qtd_duplicadas)
if qtd_duplicadas > 0:
    df = df.drop_duplicates().copy()
    print("Duplicidades removidas.")
    print("Nova dimensão:", df.shape)

Duplicidades exatas: 0


In [120]:
# Cardinalidade das variáveis categóricas
categoricas = df.select_dtypes(
 include="object").columns

cardinalidade = (
 df[categoricas]
 .nunique(dropna=True)
 .sort_values(ascending=False)
 .reset_index()
)
cardinalidade.columns = [
 "variavel","qtd_categorias"]
display(cardinalidade.head(30))

C:\Users\Aluno\AppData\Local\Temp\ipykernel_19368\1504401738.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categoricas = df.select_dtypes(


,variavel,qtd_categorias
0,latitude,69294
1,longitude,69237
2,km,7655
3,municipio,1844
4,horario,1412
5,tracado_via,605
6,uop,395
7,data_inversa,365
8,delegacia,153
9,causa_acidente,69


In [121]:
# Autor: Danilo Farias
# Copiloto: ChatGPT
# Conversão segura de valores numéricos com vírgula decimal
# Ex.: "546,2" -> 546.2

colunas_para_converter = [
 "br","km","pessoas","mortos","feridos",
 "feridos_leves","feridos_graves",
 "ilesos","ignorados","veiculos"
]

for coluna in colunas_para_converter:
    if coluna in df.columns:
        valores_originais = df[coluna]

        valores_texto = (
            valores_originais.astype("string")
            .str.strip()
            .str.replace(",", ".", regex=False)
        )

        valores_convertidos = pd.to_numeric(
            valores_texto,
            errors="coerce"
        )

        # Impede perda silenciosa de valores inválidos
        falhas = (
            valores_originais.notna()
            & valores_convertidos.isna()
        )

        if falhas.any():
            exemplos = valores_originais[falhas].head().tolist()
            raise ValueError(
                f"Falha na conversão da coluna '{coluna}'. "
                f"Exemplos: {exemplos}"
            )

        df[coluna] = valores_convertidos

display(df[colunas_para_converter].dtypes)

df.head(100)

br                  Int64
km                Float64
pessoas             Int64
mortos              Int64
feridos             Int64
feridos_leves       Int64
feridos_graves      Int64
ilesos              Int64
ignorados           Int64
veiculos            Int64
dtype: object

,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,pessoas,mortos,feridos_leves,feridos_graves,ilesos,ignorados,feridos,veiculos,latitude,longitude,regional,delegacia,uop
0,652493,01/01/2025,quarta-feira,06:20:00,SP,116,225.0,GUARULHOS,Reação tardia ou ineficiente do condutor,Tombamento,Com Vítimas Feridas,Pleno dia,Decrescente,Céu Claro,Múltipla,Reta;Declive,Sim,2,0,1,0,0,1,1,2,"-23,48586772","-46,54075317",SPRF-SP,DEL01-SP,UOP01-DEL01-SP
1,652519,01/01/2025,quarta-feira,07:50:00,CE,116,546.2,PENAFORTE,Pista esburacada,Colisão frontal,NaN,Pleno dia,Crescente,Céu Claro,Simples,Reta,Não,6,1,1,0,1,4,1,6,"-7,812288","-39,08333306",SPRF-CE,DEL05-CE,UOP03-DEL05-CE
2,652522,01/01/2025,quarta-feira,08:45:00,PR,369,88.2,CORNELIO PROCOPIO,Reação tardia ou ineficiente do condutor,Colisão traseira,Com Vítimas Feridas,Pleno dia,Crescente,Sol,Dupla,Reta;Aclive,Sim,5,0,3,0,2,0,3,2,"-23,182565","-50,637228",SPRF-PR,DEL07-PR,UOP05-DEL07-PR
3,652544,01/01/2025,quarta-feira,11:00:00,PR,116,74.0,CAMPINA GRANDE DO SUL,Reação tardia ou ineficiente do condutor,Saída de leito carroçável,Com Vítimas Feridas,Pleno dia,Crescente,Céu Claro,Dupla,Reta,Não,5,0,1,0,4,0,1,2,"-25,36517687","-49,04223028",SPRF-PR,DEL01-PR,UOP02-DEL01-PR
4,652549,01/01/2025,quarta-feira,09:30:00,MG,251,471.0,FRANCISCO SA,Velocidade Incompatível,Colisão frontal,Com Vítimas Feridas,Pleno dia,Decrescente,Chuva,Simples,Curva;Declive,Não,5,0,1,1,1,2,2,4,"-16,46801304","-43,43121303",SPRF-MG,DEL12-MG,UOP01-DEL12-MG
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,653157,04/01/2025,sábado,14:50:00,RJ,116,133.0,MAGE,Reação tardia ou ineficiente do condutor,Colisão traseira,Com Vítimas Feridas,Pleno dia,Decrescente,Sol,Dupla,Reta,Não,5,0,0,1,2,3,1,6,"-22,65418284","-43,14382995",SPRF-RJ,DEL04-RJ,UOP02-DEL04-RJ
96,653169,04/01/2025,sábado,16:35:00,MG,116,310.0,ITAMBACURI,Transitar no Acostamento,Colisão lateral mesmo sentido,Sem Vítimas,Pleno dia,Crescente,Céu Claro,Simples,Reta,Não,4,0,0,0,3,1,0,3,"-18,071493","-41,67787",SPRF-MG,DEL11-MG,UOP01-DEL11-MG
97,653176,04/01/2025,sábado,14:30:00,PB,230,162.7,CAMPINA GRANDE,Condutor deixou de manter distância do veículo...,Engavetamento,Sem Vítimas,Pleno dia,Decrescente,Sol,Simples,Reta,Não,3,0,0,0,3,0,0,3,"-7,2277","-35,9439",SPRF-PB,DEL02-PB,UOP01-DEL02-PB
98,653177,04/01/2025,sábado,16:45:00,PB,104,109.9,LAGOA SECA,Condutor deixou de manter distância do veículo...,Colisão traseira,Sem Vítimas,Pleno dia,Decrescente,Céu Claro,Simples,Reta;Aclive,Não,2,0,0,0,2,0,0,2,"-7,11997886","-35,86414665",SPRF-PB,DEL02-PB,UOP04-DEL02-PB


In [122]:
df["data_inversa"] = pd.to_datetime(df["data_inversa"],format="%d/%m/%Y", errors="coerce")
df["ano"] = df["data_inversa"].dt.year
df["mes"] = df["data_inversa"].dt.month
df["trimestre"] = df["data_inversa"].dt.quarter
df["dia_semana_num"] = df["data_inversa"].dt.dayofweek
df["fim_de_semana"] = df["dia_semana_num"].isin([5,6]).astype(int)

df.sample(100)

,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,pessoas,mortos,feridos_leves,feridos_graves,ilesos,ignorados,feridos,veiculos,latitude,longitude,regional,delegacia,uop,ano,mes,trimestre,dia_semana_num,fim_de_semana
31200,670896,2025-04-01,terça-feira,08:30:00,SP,116,54.0,LORENA,Demais falhas na via,Colisão com objeto,Sem Vítimas,Pleno dia,Decrescente,Céu Claro,Dupla,Reta,Sim,2,0,0,0,2,0,0,1,"-22,76098617","-45,12084961",SPRF-SP,DEL08-SP,UOP02-DEL08-SP,2025,4,2,1,0
45228,702941,2025-07-06,domingo,15:15:00,SP,381,79.0,SAO PAULO,Transitar na contramão,Colisão transversal,Com Vítimas Feridas,Pleno dia,Decrescente,Céu Claro,Simples,Interseção de Vias,Sim,3,0,1,0,2,0,1,2,"-23,40861","-46,58044",SPRF-SP,DEL03-SP,UOP01-DEL03-SP,2025,7,3,6,1
61742,727679,2025-10-26,domingo,14:05:00,SC,280,92.7,CORUPA,Reação tardia ou ineficiente do condutor,Saída de leito carroçável,Com Vítimas Feridas,Pleno dia,Decrescente,Chuva,Simples,Reta,Não,1,0,1,0,0,0,1,1,"-26,392457","-49,301785",SPRF-SC,DEL06-SC,UOP03-DEL06-SC,2025,10,4,6,1
53003,714670,2025-08-29,sexta-feira,02:55:00,PR,277,71.0,SAO JOSE DOS PINHAIS,Condutor Dormindo,Colisão traseira,Com Vítimas Feridas,Plena Noite,Decrescente,Nublado,Dupla,Reta,Não,3,0,1,1,1,0,2,2,"-25,50953812","-49,13277833",SPRF-PR,DEL01-PR,UOP07-DEL01-PR,2025,8,3,4,0
36995,690689,2025-05-11,domingo,20:00:00,SC,282,277.1,SAO JOSE DO CERRITO,Transitar na contramão,Colisão frontal,Com Vítimas Feridas,Plena Noite,Crescente,Céu Claro,Simples,Curva,Não,4,0,2,1,0,1,3,2,"-27,58873157","-50,73526595",SPRF-SC,DEL05-SC,UOP02-DEL05-SC,2025,5,2,6,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23313,658996,2025-02-03,segunda-feira,06:30:00,MA,316,291.6,PIO XII,Pista esburacada,Saída de leito carroçável,Com Vítimas Feridas,Pleno dia,Crescente,Céu Claro,Simples,Reta,Não,2,0,1,0,1,0,1,1,"-3,82923","-45,190002",SPRF-MA,DEL02-MA,UOP01-DEL02-MA,2025,2,1,0,0
15248,728806,2025-10-31,sexta-feira,17:10:00,GO,20,100.0,VILA BOA,Acostamento em desnível,Tombamento,Sem Vítimas,Pleno dia,Decrescente,Nublado,Simples,Reta,Não,1,0,0,0,0,1,0,1,"-14,93944152","-46,98473644",SPRF-DF,DEL02-DF,UOP02-DEL02-DF,2025,10,4,4,0
17762,739373,2025-12-16,terça-feira,18:25:00,SC,101,463.0,PASSO DE TORRES,Manobra de mudança de faixa,Colisão lateral mesmo sentido,Com Vítimas Feridas,Pleno dia,Decrescente,Céu Claro,Dupla,Reta,Não,4,0,0,1,2,1,1,3,"-29,28963143","-49,75819437",SPRF-RS,DEL03-RS,UOP02-DEL03-RS,2025,12,4,1,0
10676,709698,2025-08-06,quarta-feira,06:20:00,SP,153,306.9,CAMPOS NOVOS PAULISTA,Reação tardia ou ineficiente do condutor,Colisão traseira,Com Vítimas Feridas,Amanhecer,Crescente,Céu Claro,Simples,Reta;Aclive;Interseção de Vias,Não,31,0,2,0,29,0,2,2,"-22,6134422","-49,8923325",SPRF-SP,DEL07-SP,UOP03-DEL07-SP,2025,8,3,2,0


In [123]:
horario_limpo = df["horario"].astype(str).str.strip()
df["hora"] = pd.to_datetime(
 horario_limpo, format="%H:%M:%S",
 errors="coerce").dt.hour

def classificar_turno(hora):
    if pd.isna(hora): return "IGNORADO"
    if 0 <= hora <= 5: return "MADRUGADA"
    if 6 <= hora <= 11: return "MANHA"
    if 12 <= hora <= 17: return "TARDE"
    return "NOITE"

df["turno"] = df["hora"].apply(classificar_turno)

df.head()

,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,pessoas,mortos,feridos_leves,feridos_graves,ilesos,ignorados,feridos,veiculos,latitude,longitude,regional,delegacia,uop,ano,mes,trimestre,dia_semana_num,fim_de_semana,hora,turno
0,652493,2025-01-01,quarta-feira,06:20:00,SP,116,225.0,GUARULHOS,Reação tardia ou ineficiente do condutor,Tombamento,Com Vítimas Feridas,Pleno dia,Decrescente,Céu Claro,Múltipla,Reta;Declive,Sim,2,0,1,0,0,1,1,2,"-23,48586772","-46,54075317",SPRF-SP,DEL01-SP,UOP01-DEL01-SP,2025,1,1,2,0,6,MANHA
1,652519,2025-01-01,quarta-feira,07:50:00,CE,116,546.2,PENAFORTE,Pista esburacada,Colisão frontal,NaN,Pleno dia,Crescente,Céu Claro,Simples,Reta,Não,6,1,1,0,1,4,1,6,"-7,812288","-39,08333306",SPRF-CE,DEL05-CE,UOP03-DEL05-CE,2025,1,1,2,0,7,MANHA
2,652522,2025-01-01,quarta-feira,08:45:00,PR,369,88.2,CORNELIO PROCOPIO,Reação tardia ou ineficiente do condutor,Colisão traseira,Com Vítimas Feridas,Pleno dia,Crescente,Sol,Dupla,Reta;Aclive,Sim,5,0,3,0,2,0,3,2,"-23,182565","-50,637228",SPRF-PR,DEL07-PR,UOP05-DEL07-PR,2025,1,1,2,0,8,MANHA
3,652544,2025-01-01,quarta-feira,11:00:00,PR,116,74.0,CAMPINA GRANDE DO SUL,Reação tardia ou ineficiente do condutor,Saída de leito carroçável,Com Vítimas Feridas,Pleno dia,Crescente,Céu Claro,Dupla,Reta,Não,5,0,1,0,4,0,1,2,"-25,36517687","-49,04223028",SPRF-PR,DEL01-PR,UOP02-DEL01-PR,2025,1,1,2,0,11,MANHA
4,652549,2025-01-01,quarta-feira,09:30:00,MG,251,471.0,FRANCISCO SA,Velocidade Incompatível,Colisão frontal,Com Vítimas Feridas,Pleno dia,Decrescente,Chuva,Simples,Curva;Declive,Não,5,0,1,1,1,2,2,4,"-16,46801304","-43,43121303",SPRF-MG,DEL12-MG,UOP01-DEL12-MG,2025,1,1,2,0,9,MANHA


In [124]:
def criar_faixa_horaria(hora):
    if pd.isna(hora):
        return "IGNORADO"
    inicio = int(hora // 3) * 3
    fim = inicio + 2
    return f"{inicio:02d}h-{fim:02d}h"

df["faixa_horaria"] = df["hora"].apply(criar_faixa_horaria)

display(df["faixa_horaria"].value_counts(dropna=False).sort_index())

faixa_horaria
00h-02h     3959
03h-05h     4948
06h-08h    11517
09h-11h     9342
12h-14h     9678
15h-17h    12624
18h-20h    13473
21h-23h     6988
Name: count, dtype: int64

In [125]:
# Autor: Danilo Farias
# Copiloto: ChatGPT
# Limpeza das colunas textuais
colunas_string = df.select_dtypes(
    include=["object", "string"]
).columns

for coluna in colunas_string:
    df[coluna] = (
        df[coluna]
        .astype("string")
        .str.strip()
        .str.upper()
        .replace({
            "": pd.NA,
            "NAN": pd.NA,
            "NULL": pd.NA
        })
    )

display(df[colunas_string].head())

,dia_semana,horario,uf,municipio,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,latitude,longitude,regional,delegacia,uop,turno,faixa_horaria
0,QUARTA-FEIRA,06:20:00,SP,GUARULHOS,REAÇÃO TARDIA OU INEFICIENTE DO CONDUTOR,TOMBAMENTO,COM VÍTIMAS FERIDAS,PLENO DIA,DECRESCENTE,CÉU CLARO,MÚLTIPLA,RETA;DECLIVE,SIM,"-23,48586772","-46,54075317",SPRF-SP,DEL01-SP,UOP01-DEL01-SP,MANHA,06H-08H
1,QUARTA-FEIRA,07:50:00,CE,PENAFORTE,PISTA ESBURACADA,COLISÃO FRONTAL,<NA>,PLENO DIA,CRESCENTE,CÉU CLARO,SIMPLES,RETA,NÃO,"-7,812288","-39,08333306",SPRF-CE,DEL05-CE,UOP03-DEL05-CE,MANHA,06H-08H
2,QUARTA-FEIRA,08:45:00,PR,CORNELIO PROCOPIO,REAÇÃO TARDIA OU INEFICIENTE DO CONDUTOR,COLISÃO TRASEIRA,COM VÍTIMAS FERIDAS,PLENO DIA,CRESCENTE,SOL,DUPLA,RETA;ACLIVE,SIM,"-23,182565","-50,637228",SPRF-PR,DEL07-PR,UOP05-DEL07-PR,MANHA,06H-08H
3,QUARTA-FEIRA,11:00:00,PR,CAMPINA GRANDE DO SUL,REAÇÃO TARDIA OU INEFICIENTE DO CONDUTOR,SAÍDA DE LEITO CARROÇÁVEL,COM VÍTIMAS FERIDAS,PLENO DIA,CRESCENTE,CÉU CLARO,DUPLA,RETA,NÃO,"-25,36517687","-49,04223028",SPRF-PR,DEL01-PR,UOP02-DEL01-PR,MANHA,09H-11H
4,QUARTA-FEIRA,09:30:00,MG,FRANCISCO SA,VELOCIDADE INCOMPATÍVEL,COLISÃO FRONTAL,COM VÍTIMAS FERIDAS,PLENO DIA,DECRESCENTE,CHUVA,SIMPLES,CURVA;DECLIVE,NÃO,"-16,46801304","-43,43121303",SPRF-MG,DEL12-MG,UOP01-DEL12-MG,MANHA,09H-11H


In [126]:
categoricas_importantes = ["uf","municipio","causa_acidente","tipo_acidente","fase_dia","condicao_metereologica","tipo_pista","tracado_via","uso_solo","classificacao_acidente","dia_semana"]
print("\nAntes\n")
print(df[categoricas_importantes]
.isna().sum()
.sort_values(ascending=False))

for coluna in categoricas_importantes:
    if coluna in df.columns:
        df[coluna] = df[coluna].fillna("IGNORADO")
print("\nDepois\n")
print(df[categoricas_importantes]
.isna().sum()
.sort_values(ascending=False))


Antes

classificacao_acidente    1
municipio                 0
uf                        0
causa_acidente            0
tipo_acidente             0
condicao_metereologica    0
fase_dia                  0
tipo_pista                0
tracado_via               0
uso_solo                  0
dia_semana                0
dtype: int64

Depois

uf                        0
municipio                 0
causa_acidente            0
tipo_acidente             0
fase_dia                  0
condicao_metereologica    0
tipo_pista                0
tracado_via               0
uso_solo                  0
classificacao_acidente    0
dia_semana                0
dtype: int64


In [127]:
contagens_vitimas = ["mortos","feridos","feridos_leves",
 "feridos_graves","pessoas","veiculos"]

for coluna in contagens_vitimas:
    if coluna in df.columns:
        df[coluna] = df[coluna].fillna(0)

print(df[[c for c in contagens_vitimas
 if c in df.columns]].isna().sum())

mortos            0
feridos           0
feridos_leves     0
feridos_graves    0
pessoas           0
veiculos          0
dtype: int64


In [128]:
df["acidente_fatal"] = np.where(
    df["mortos"] >= 1, 1, 0)

validacao_alvo = (
  df["acidente_fatal"]
  .value_counts(dropna=False)
  .rename_axis("acidente_fatal")
  .reset_index(name="qtd"))
validacao_alvo["perc"] = (
  validacao_alvo["qtd"] /
  validacao_alvo["qtd"].sum() * 100)
display(validacao_alvo)

,acidente_fatal,qtd,perc
0,0,67319,92.816666
1,1,5210,7.183334
